# 260403 Hybrid Search & Query Expansion

**4주차 Day 4** - 하이브리드 검색(키워드 + 벡터)과 쿼리 확장 기법

어제까지 벡터 검색(임베딩 기반)만으로는 한계가 있다는 것을 배웠다.  
오늘은 키워드 검색과 벡터 검색을 **합치는 방법(Hybrid Search)**, 그리고 검색 쿼리 자체를 **확장하는 기법(Query Expansion)**을 다룬다.

| 주제 | 비유 |
|------|------|
| Hybrid Search | 도서관에서 제목 검색(키워드) + 내용 유사도 검색(벡터)을 동시에 돌리는 것 |
| RRF | 두 심사위원의 점수 기준이 다를 때, 점수 대신 **순위**로 합산하는 방법 |
| Multi-Query | 하나의 질문을 여러 방식으로 바꿔서 검색 범위를 넓히는 것 |
| HyDE | 질문에 대한 '가짜 답변'을 먼저 만들고, 그걸로 검색하는 트릭 |

In [7]:
# 벡터 검색(임베딩 기반)만으로는 한계
# - 벡터 검색은 의미는 잘 잡지만 정확한 키워드를 놓칠 수 있다.
# 질문: "BM25 알고리즘"
# 벡터 검색: "TF-IDF 기반 검색" (의미적으로 비슷한 문서를 가져옴)
# → 정작 "BM25"가 정확히 적힌 문서를 놓침
# 고유명사, 전문용어, 약어처럼 그 단어 자체가 중요한 경우에 약합니다. 그래서 키워드 검색(BM25)과 섞는 Hybrid Search가 필요한 이유.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w4_rag_evaluation/llm_260403_hybrid_search.ipynb)

## 0. Setup

In [9]:
# Colab 환경이면 아래 주석 해제
!pip install -q openai langchain langchain-openai langchain-community langchain-classic faiss-cpu python-dotenv sentence-transformers kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 whi

In [11]:
# --- Colab Secrets 사용 시 ---
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# --- 로컬 .env 사용 시 ---
import os
import math
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv

# load_dotenv()

# 임베딩 모델 초기화 (1536차원 벡터 생성)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 실습 데이터 준비

10개의 짧은 한국어 문서를 준비한다.  
각 문서는 AI/ML 관련 주제를 한 문장으로 설명한다.

In [12]:
documents = [
    "Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.",
    "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.",
    "GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.",
    "RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.",
    "FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워크입니다.",
    "트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.",
    "FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러리입니다.",
    "프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.",
    "임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.",
]

## 2. 임베딩 생성 비교: OpenAI vs SentenceTransformer

두 가지 임베딩 모델을 비교해 본다.  
- **OpenAI text-embedding-3-small**: 1536차원 (API 호출, 유료)  
- **all-MiniLM-L6-v2**: 384차원 (로컬 실행, 무료)  
비유: 같은 문장을 두 명의 번역가가 각각 다른 언어로 번역하는 것. 차원이 다르지만 의미는 보존된다.

In [13]:
# OpenAI 임베딩: 10개 문서 -> (10, 1536) 행렬
doc_embeddings = embeddings.embed_documents(documents)
print(f"OpenAI 임베딩: {len(doc_embeddings)}개 문서, 각 {len(doc_embeddings[0])}차원")
print(np.array(doc_embeddings).shape)

OpenAI 임베딩: 10개 문서, 각 1536차원
(10, 1536)


In [14]:
# SentenceTransformer 임베딩: 로컬에서 실행, 384차원
# !pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding = embedding_model.encode(documents)
print(f"SentenceTransformer 임베딩: {embedding.shape}")
# (10, 384) -- OpenAI보다 차원이 낮지만 로컬에서 무료로 사용 가능

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer 임베딩: (10, 384)


## 3. 키워드 검색 vs 벡터 검색

두 검색 방식의 차이를 직접 비교한다.  
- **키워드 검색**: 단어가 정확히 일치해야 함 ("파이썬" != "Python")  
- **벡터 검색**: 의미적으로 유사하면 매칭됨 ("파이썬" ~ "Python")

In [15]:
def keyword_search(query, docs, top_k=3):
    """단순 키워드 검색: 쿼리와 문서의 단어 겹침(overlap) 개수로 점수 매김"""
    query_tokens = set(query.lower().split())  # 쿼리를 소문자로 바꾸고 단어 단위로 분리
    scores = []
    for i, doc in enumerate(docs):
        doc_tokens = set(doc.lower().split())  # 문서도 동일하게 처리
        overlap = len(query_tokens & doc_tokens)  # 교집합 = 겹치는 단어 수
        scores.append((i, overlap))

    # 겹침 수가 많은 순서대로 정렬
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# 테스트: "Python"이라는 단어가 정확히 있는 문서를 찾는다
results = keyword_search("Python 프로그래밍 언어", documents)
print("=== 키워드 검색 결과 ===")
for idx, score in results:
    print(f"[{idx}] overlap = {score} | {documents[idx][:40]}")

=== 키워드 검색 결과 ===
[0] overlap = 1 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
[3] overlap = 1 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행
[1] overlap = 0 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니


In [16]:
def vector_search(query, docs, doc_embs, top_k=3):
    """벡터(의미) 검색: 코사인 유사도로 의미적 유사성 측정"""
    q_emb = np.array(embeddings.embed_query(query))  # 쿼리를 벡터로 변환
    # 코사인 유사도 = (A . B) / (|A| * |B|)
    # 비유: 두 화살표가 같은 방향을 가리킬수록 유사도가 높다
    similarities = np.dot(doc_embs, q_emb) / (np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(q_emb))
    top_indices = similarities.argsort()[::-1][:top_k]  # 유사도 높은 순
    return [(i, similarities[i]) for i in top_indices]

results = vector_search("Python 프로그래밍 언어", documents, np.array(doc_embeddings), top_k=3)
print("=== 벡터 검색 결과 ===")
for idx, score in results:
    print(f"[{idx}] similarity = {score:.4f} | {documents[idx][:40]}")

=== 벡터 검색 결과 ===
[0] similarity = 0.5528 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
[1] similarity = 0.3718 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니
[3] similarity = 0.3604 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행


In [17]:
# 키워드 검색이 실패하는 케이스: 'FAISS'라는 고유명사
# 키워드로는 정확히 매칭되지만, 벡터 검색은 의미적으로 관련 문서도 함께 찾아줌
results = vector_search('FAISS', documents, np.array(doc_embeddings), top_k=3)
print("=== 'FAISS' 벡터 검색 ===")
for idx, score in results:
    print(f"[{idx}] similarity = {score:.4f} | {documents[idx][:40]}")

=== 'FAISS' 벡터 검색 ===
[7] similarity = 0.5279 | FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러
[4] similarity = 0.1923 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다
[5] similarity = 0.0964 | FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워


## 4. 두 검색 결과의 겹침(Overlap) 분석

키워드 검색과 벡터 검색이 같은 문서를 찾는 비율을 측정한다.  
겹침이 낮다 = 두 검색이 서로 다른 문서를 찾고 있다 = **합치면 더 다양한 결과**를 기대할 수 있다.

In [18]:
def overlap_rate(keyword_results, vector_results):
    """두 검색 결과의 겹침 비율 (Jaccard 유사도와 같은 개념)"""
    kw_ids = set(idx for idx, _ in keyword_results)
    vec_ids = set(idx for idx, _ in vector_results)

    overlap = kw_ids & vec_ids   # 교집합: 둘 다 찾은 문서
    union = kw_ids | vec_ids     # 합집합: 하나라도 찾은 문서
    return len(overlap) / len(union)

# 쿼리별로 겹침 비율 확인
for query in ['Python 프로그래밍', '딥러닝 모델', 'FAISS 라이브러리']:
    kw = keyword_search(query, documents, top_k=5)
    vec = vector_search(query, documents, np.array(doc_embeddings), top_k=5)
    rate = overlap_rate(kw, vec)
    print(f"{query} -> overlap: {rate:.2f}")
    # overlap이 낮을수록 두 검색이 보완적 -> 하이브리드 검색의 가치가 높아짐

Python 프로그래밍 -> overlap: 0.67
딥러닝 모델 -> overlap: 0.43
FAISS 라이브러리 -> overlap: 0.25


## 5. Simple Hybrid Search (점수 기반 합산)

두 검색 점수를 정규화(0~1)해서 단순 합산하는 방식.  
비유: 영어 시험 100점 만점 + 수학 시험 50점 만점을 합산하려면, 먼저 둘 다 100점 만점으로 **환산**해야 공정하다.

In [19]:
def simple_hybrid(query, docs, doc_embs, top_k=3):
    """키워드 + 벡터 점수를 정규화 후 합산하는 단순 하이브리드 검색"""
    # 모든 문서에 대해 점수를 구함 (top_k=len(docs))
    kw = keyword_search(query, docs, top_k=len(docs))
    vec = vector_search(query, docs, np.array(doc_embs), top_k=len(docs))

    kw_scores = {idx: score for idx, score in kw}
    vec_scores = {idx: score for idx, score in vec}

    # 최대값으로 나눠서 0~1 사이로 정규화 (Min-Max 스케일링의 간소화 버전)
    kw_max = max(kw_scores.values()) or 1
    vec_max = max(vec_scores.values()) or 1

    combined = {}
    for idx in range(len(docs)):
        kw_score = kw_scores.get(idx, 0) / kw_max    # 키워드 점수 정규화
        vec_score = vec_scores.get(idx, 0) / vec_max  # 벡터 점수 정규화
        combined[idx] = kw_score + vec_score  # 단순 합산 (최대 2.0)

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# 여러 쿼리에 대해 하이브리드 검색 결과 확인
for query in ['Python 프로그래밍 언어', '딥러닝 모델 구조', 'FAISS']:
    results = simple_hybrid(query, documents, np.array(doc_embeddings), top_k=3)
    print(f"\n[{query}]")
    for idx, score in results:
        print(f"  [{idx}] {score:.4f} | {documents[idx][:40]}")


[Python 프로그래밍 언어]
  [0] 2.0000 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
  [3] 1.6520 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행
  [1] 0.6725 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니

[딥러닝 모델 구조]
  [6] 2.0000 | 트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.
  [3] 0.4905 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행
  [0] 0.4190 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니

[FAISS]
  [7] 1.0000 | FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러
  [4] 0.3644 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다
  [5] 0.1827 | FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워


## 6. TF-IDF

키워드 검색의 고전적 개선 방법.  
- **TF (Term Frequency)**: 단어가 문서에 자주 나올수록 중요  
- **IDF (Inverse Document Frequency)**: 여러 문서에 공통으로 나오는 단어는 덜 중요  

비유: "있다", "이다" 같은 흔한 단어(TF 높아도 IDF 낮음)보다 "트랜스포머"같은 특수한 단어(IDF 높음)가 검색에 더 유용하다.

In [20]:
class TFIDF:
    def __init__(self, documents):
        self.docs = documents
        self.tokenized = [doc.lower().split() for doc in documents]  # 소문자 + 단어 분리
        self.N = len(documents)  # 전체 문서 수
        # DF(Document Frequency): 각 단어가 등장하는 문서 수
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):  # set으로 중복 제거 (한 문서에서 여러 번 나와도 1번으로 카운트)
                self.df[t] = self.df.get(t, 0) + 1

    def tf(self, term, doc_tokens):
        """TF = (단어 등장 횟수) / (문서 전체 단어 수)"""
        return doc_tokens.count(term) / len(doc_tokens)

    def idf(self, term):
        """IDF = log(전체 문서 수 / 단어가 등장한 문서 수)
        흔한 단어일수록 값이 작고, 희귀한 단어일수록 값이 크다"""
        return math.log(self.N / self.df.get(term, 1))

    def score(self, query, doc_idx):
        """쿼리의 각 단어에 대해 TF * IDF를 합산"""
        tokens = self.tokenized[doc_idx]
        return sum(self.tf(t, tokens) * self.idf(t) for t in query.lower().split())

    def search(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

tfidf = TFIDF(documents)
print("=== TF-IDF 검색 ===")
for idx, score in tfidf.search('Python 프로그래밍'):
    print(f"[{idx}] {score:.4f} | {documents[idx][:40]}")

=== TF-IDF 검색 ===
[0] 0.2878 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
[1] 0.0000 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니
[2] 0.0000 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니


## 7. BM25 (TF-IDF의 개선판)

TF-IDF의 두 가지 문제를 **k1, b** 파라미터로 보완한다.  
- **k1**: TF 값에 상한선을 둠 (같은 단어가 100번 나와도 무한히 올라가지 않음)  
- **b**: 문서 길이를 보정 (긴 문서가 불리하지 않도록)  

비유: TF-IDF가 단순한 체중계라면, BM25는 체지방률도 고려하는 체성분 분석기.

In [21]:
class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1, self.b = k1, b  # k1: TF 포화도, b: 문서 길이 보정 강도
        self.docs = documents
        self.tokenized = [doc.lower().split() for doc in documents]
        self.N = len(documents)
        self.avgdl = sum(len(d) for d in self.tokenized) / self.N  # 평균 문서 길이
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):
                self.df[t] = self.df.get(t, 0) + 1

    def idf(self, term):
        """BM25용 IDF: log((N - df + 0.5) / (df + 0.5) + 1)
        TF-IDF의 IDF보다 안정적 (음수가 나오지 않도록 +1)"""
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query, doc_idx):
        tokens = self.tokenized[doc_idx]
        dl = len(tokens)  # 현재 문서 길이
        tf_counter = Counter(tokens)
        total = 0.0
        for t in query.lower().split():
            tf = tf_counter.get(t, 0)
            # 핵심 수식: TF를 k1으로 상한 제한, b로 문서 길이 보정
            numerator = tf * (self.k1 + 1)
            denominator = tf + (self.k1 * (1 - self.b + self.b * dl / self.avgdl))
            total += self.idf(t) * numerator / denominator
        return total

    def search(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

bm25 = BM25(documents)
print("=== BM25 검색 ===")
for idx, score in bm25.search('Python 프로그래밍'):
    print(f"[{idx}] {score:.4f} | {documents[idx][:40]}")

=== BM25 검색 ===
[0] 2.0361 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
[1] 0.0000 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니
[2] 0.0000 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니


## 8. LangChain Retriever로 간편하게 사용

위에서 직접 구현한 BM25, FAISS를 **LangChain 래퍼**로 더 쉽게 사용한다.  
EnsembleRetriever로 두 검색기를 합치면 가중치(weights)만 조절하면 된다.

In [22]:
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Document 객체로 변환 (LangChain이 요구하는 형태)
docs_lc = [Document(page_content=d, metadata={"index": i}) for i, d in enumerate(documents)]

# BM25 리트리버 생성
bm25_retriever = BM25Retriever.from_documents(docs_lc)
bm25_retriever.k = 3  # 상위 3개만 반환

# FAISS 벡터 리트리버 생성
vectorstore = FAISS.from_documents(docs_lc, embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

ImportError: Could not import rank_bm25, please install with `pip install rank_bm25`.

In [ ]:
# BM25 검색
print("=== BM25 Retriever ===")
results = bm25_retriever.invoke('Python 프로그래밍')
for r in results:
    print(f"  [{r.metadata['index']}] {r.page_content[:40]}")

# 벡터 검색
print("\n=== Vector Retriever ===")
results = vector_retriever.invoke('딥러닝 모델 구조')
for r in results:
    print(f"  [{r.metadata['index']}] {r.page_content[:40]}")

In [ ]:
# EnsembleRetriever: 두 검색기를 가중치로 합침
from langchain_classic.retrievers import EnsembleRetriever

ensemble = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5],  # 벡터 50%, 키워드 50%
)

results = ensemble.invoke("Python 데이터 과학")
print("=== Ensemble (0.5/0.5) ===")
for r in results:
    print(f"  [{r.metadata['index']}] {r.page_content[:40]}")

In [ ]:
# 가중치를 바꿔보면서 결과 변화 확인
for w_vec, w_bm25 in [(0.2, 0.8), (0.5, 0.5), (0.8, 0.2)]:
    ensemble = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[w_vec, w_bm25],
    )
    results = ensemble.invoke("Python 데이터 과학")
    top_idx = results[0].metadata['index']
    print(f"BM25={w_bm25}, Vector={w_vec} -> Top-1: [{top_idx}] {results[0].page_content[:30]}")

In [ ]:
# 3가지 검색 방식 비교: BM25 vs Vector vs Ensemble
queries = ["Python 프로그래밍 언어", "딥러닝 모델 구조", "FAISS 라이브러리"]
for q in queries:
    bm25_res = bm25_retriever.invoke(q)
    vec_res = vector_retriever.invoke(q)
    ens_res = ensemble.invoke(q)

    bm25_ids = [d.metadata['index'] for d in bm25_res]
    vec_ids = [d.metadata['index'] for d in vec_res]
    ens_ids = [d.metadata['index'] for d in ens_res]

    print(f"{q}")
    print(f"  BM25: {bm25_ids}, Vector: {vec_ids}, Ensemble: {ens_ids}\n")

## 9. RRF (Reciprocal Rank Fusion) - 순위 기반 합산

**핵심 문제**: BM25 점수(수십~수백)와 코사인 유사도(-1~1)는 스케일이 완전히 다르다.  
**해결**: 점수 대신 **순위(rank)**를 사용해 합산하면 스케일 문제가 사라진다.  

수식: `RRF(d) = sum( 1 / (k + rank) )`  

- **k**가 작으면 (예: k=1): 1등과 2등의 차이가 큼 (1/2 vs 1/3)  
- **k**가 크면 (예: k=100): 1등과 2등의 차이가 미미 (1/101 vs 1/102)  

비유: temperature처럼 k는 순위 간 격차를 조절하는 '스무딩' 역할. k가 커지면 순위 차이가 평탄해진다.

In [ ]:
def rrf(rankings, k=60):
    """Reciprocal Rank Fusion: 여러 랭킹 결과를 순위의 역수로 합산
    - rankings: [(doc_id, score), ...] 형태의 리스트 여러 개
    - k: 스무딩 파라미터 (기본값 60, Elasticsearch에서도 사용)
    """
    rrf_scores = {}
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking, start=1):  # 1등부터 시작
            # 점수가 아닌 순위만 사용하므로 스케일 문제 없음
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank)

    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

# BM25와 TF-IDF 결과를 RRF로 합치기
query = 'Python 데이터 과학'
bm25_results = bm25.search(query, top_k=5)
tfidf_results = tfidf.search(query, top_k=5)

combined = rrf([bm25_results, tfidf_results], k=60)
print("=== RRF 결과 (BM25 + TF-IDF) ===")
for doc_id, score in combined[:3]:
    print(f"  [{doc_id}] {score:.6f} | {documents[doc_id][:40]}")

In [ ]:
# RRF 실습: BM25 + 벡터 검색을 RRF로 합치기
query = '대규모 언어 모델'
bm25_r = bm25.search(query, top_k=3)
vector_r = vector_search(query, documents, np.array(doc_embeddings), top_k=3)

# 각 검색기 결과 비교
print("=== BM25 결과 ===")
for rank, (idx, score) in enumerate(bm25_r, 1):
    print(f"  {rank}위: [{idx}] {documents[idx][:30]} (score={score:.4f})")

print("\n=== Vector 결과 ===")
for rank, (idx, score) in enumerate(vector_r, 1):
    print(f"  {rank}위: [{idx}] {documents[idx][:30]} (score={score:.4f})")

# RRF로 합치면 순위만 보므로 점수 스케일 차이가 문제되지 않음
combined = rrf([bm25_r, vector_r], k=60)
print("\n=== RRF 결과 ===")
for doc_id, score in combined[:3]:
    print(f"  [{doc_id}] {score:.6f} | {documents[doc_id][:30]}")

## 10. Grid Search로 최적 가중치 찾기

검색 메트릭(Precision@K, Recall@K, MRR)을 정의하고,  
가중치를 0.0~1.0까지 바꿔가며 **최적의 BM25 vs Vector 비율**을 찾는다.  

비유: 카레 만들 때 카레 가루와 물의 비율을 10번 바꿔보면서 가장 맛있는 비율을 찾는 것 = Grid Search.

In [ ]:
# 평가 데이터셋: 각 쿼리에 대한 '정답' 문서 ID
eval_dataset = [
    {"query": "Python 프로그래밍 언어", "relevant": [0, 5]},
    {"query": "자연어 처리 NLP 기술", "relevant": [1]},
    {"query": "벡터 데이터베이스", "relevant": [2, 7]},
    {"query": "GPT 대규모 언어 모델", "relevant": [3]},
    {"query": "RAG 검색 증강 생성", "relevant": [4]},
    {"query": "트랜스포머 어텐션", "relevant": [6]},
    {"query": "임베딩 벡터 변환", "relevant": [9]},
]

In [ ]:
# 검색 평가 메트릭 (w4 d2~d3에서 배운 것 복습)
def precision_at_k(retrieved, relevant, k):
    """상위 K개 중 정답 비율. 비유: 낚은 물고기 중 먹을 수 있는 비율"""
    top_k = retrieved[:k]
    return len(set(top_k) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    """정답 중 상위 K개에 포함된 비율. 비유: 먹을 수 있는 물고기 중 실제로 낚은 비율"""
    top_k = retrieved[:k]
    return len(set(top_k) & set(relevant)) / len(relevant) if relevant else 0

def mrr(retrieved, relevant):
    """첫 번째 정답이 몇 번째에 나왔는지의 역수. 1등에 정답 -> 1.0, 3등에 정답 -> 0.33"""
    for rank, doc_id in enumerate(retrieved, 1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

# 예시: 검색 결과 [3,1,4,0,2]에서 정답이 [4,0]일 때
retrieved = [3, 1, 4, 0, 2]
relevant = [4, 0]
print(f"P@3={precision_at_k(retrieved, relevant, 3):.4f}")
print(f"R@3={recall_at_k(retrieved, relevant, 3):.4f}")
print(f"MRR={mrr(retrieved, relevant):.4f}")

In [ ]:
def hybrid_search(query, w_bm25=0.5, top_k=5):
    """RRF 기반 하이브리드 검색 (BM25 가중치 조절 가능)"""
    bm25_r = bm25.search(query, top_k=top_k)
    vec_r = vector_search(query, documents, np.array(doc_embeddings), top_k=top_k)

    rrf_scores = {}
    # BM25 결과에 w_bm25 가중치 적용
    for rank, (doc_id, _) in enumerate(bm25_r, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + w_bm25 / (60 + rank)
    # 벡터 결과에 (1 - w_bm25) 가중치 적용
    for rank, (doc_id, _) in enumerate(vec_r, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1 - w_bm25) / (60 + rank)

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in ranked[:top_k]]

In [ ]:
# Grid Search: w_bm25를 0.0~1.0까지 0.1씩 바꿔가며 최적값 찾기
weights = np.arange(0, 1.05, 0.1)

results = []
for w in weights:
    mrr_sum = 0
    for item in eval_dataset:
        retrieved = hybrid_search(item['query'], w_bm25=w, top_k=5)
        mrr_sum += mrr(retrieved, item['relevant'])
    avg_mrr = mrr_sum / len(eval_dataset)
    results.append((w, avg_mrr))
    print(f"  w_bm25={w:.1f}: MRR={avg_mrr:.4f}")

# 최적 가중치 출력
best_w, best_mrr = max(results, key=lambda x: x[1])
print(f"\n>>> 최적 가중치: w_bm25={best_w:.1f} (MRR={best_mrr:.4f})")
# 이 데이터에서는 벡터 검색이 압도적이지만, 실제 대규모 데이터에서는 하이브리드가 더 나은 경우가 많다

## 11. 한국어 형태소 분석 (Kiwi)

한국어 키워드 검색의 문제: "파이썬 뭔가요?" 에서 '파이썬'과 'Python'이 매칭되지 않음.  
조사("은/는/이/가")도 분리해야 제대로 매칭됨.  

**Kiwi** 형태소 분석기로 명사(NNP, NNG)만 추출하면 키워드 검색 품질이 올라간다.

In [ ]:
# !pip install -q kiwipiepy
from kiwipiepy import Kiwi

kiwi = Kiwi()

# 형태소 분석 예시
query = '자연어 처리는 어렵습니다'
tokens = kiwi.tokenize(query)
print("전체 토큰:")
for t in tokens:
    print(f"  {t.form} ({t.tag})")

# 명사만 추출 (NNP: 고유명사, NNG: 일반명사)
print("\n명사만 추출:")
for t in tokens:
    if t.tag in ['NNP', 'NNG']:
        print(f"  {t.form}")
# 이렇게 명사만 뽑아서 키워드 검색에 사용하면 조사 문제 해결

In [ ]:
# 고유명사 인식 테스트
query = '이순신 장군'
tokens = kiwi.tokenize(query)
print([f"{t.form}({t.tag})" for t in tokens])
# '이순신'을 NNP(고유명사)로 인식, '장군'을 NNG(일반명사)로 인식

## 12. Query Expansion - 쿼리 확장 기법

검색 쿼리를 그대로 쓰지 말고, **LLM으로 확장/변형**해서 검색 품질을 올리는 방법.  
3가지 주요 기법:

| 기법 | 방법 | 비유 |
|------|------|------|
| **Multi-Query** | 하나의 질문을 여러 관점으로 재작성 | 같은 질문을 다르게 표현해서 검색 |
| **HyDE** | 질문의 가상 답변을 생성 -> 답변으로 검색 | 정답을 미리 짐작해서 비슷한 문서 찾기 |
| **Query Decomposition** | 복잡한 질문을 여러 sub-query로 분해 | CoT처럼 단계별로 나눠서 검색 |

In [ ]:
# 쿼리 확장 실습을 위한 데이터 (문서 12개로 확장)
documents = [
    "Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.",
    "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.",
    "GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.",
    "RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.",
    "FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워크입니다.",
    "트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.",
    "FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러리입니다.",
    "프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.",
    "임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.",
    "LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다.",
    "청킹(Chunking)은 긴 문서를 검색에 적합한 크기로 분할하는 기법입니다.",
]

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# FAISS 벡터 스토어 재구축
docs = [Document(page_content=text, metadata={'doc_id': i}) for i, text in enumerate(documents)]
vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
# 유틸리티 함수: FAISS의 distance를 유사도 score로 변환
def _score(distance):
    """FAISS는 거리(distance) 반환 -> 역수로 유사도(score)로 변환
    거리가 0에 가까울수록 유사도가 1에 가깝다"""
    return 1.0 / (1.0 + float(distance))

def search_faiss(query, top_k=3):
    """FAISS 텍스트 쿼리 검색"""
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc.metadata['doc_id'], _score(dist)) for doc, dist in results]

def search_by_vector(embedding, top_k=3):
    """FAISS 벡터 직접 검색 (HyDE에서 사용)"""
    results = vectorstore.similarity_search_with_score_by_vector(embedding, k=top_k)
    return [(doc.metadata['doc_id'], _score(dist)) for doc, dist in results]

# 테스트
print("=== FAISS 검색: 'Python 프로그래밍' ===")
results = search_faiss('Python 프로그래밍')
for idx, score in results:
    print(f"  [{idx}] {score:.4f} | {documents[idx][:40]}")

### 12-1. Multi-Query Retrieval

LLM에게 원래 질문을 **3가지 다른 관점**으로 다시 작성하게 한다.  
각 쿼리로 검색 -> RRF로 합산하면 더 다양한 문서를 찾을 수 있다.

주의: 너무 많으면 오히려 노이즈. 연구에 의하면 **3개가 가장 효율적**이라고 함.

In [ ]:
# 수동 쿼리 확장 먼저 시연
query = "AI 챗봇 답변 품질 높이기"
print(f"원래 쿼리: {query}")
print("\n=== 원래 쿼리로만 검색 ===")
results = search_faiss(query, top_k=3)
for idx, score in results:
    print(f"  [{idx}] {score:.4f} | {documents[idx][:40]}")

# 수동으로 쿼리 확장
expanded_queries = [
    "AI 챗봇 답변 품질 높이기",       # 원래 쿼리
    "RAG 검색 증강 생성 기법",         # 전문 용어로 확장
    "프롬프트 엔지니어링 LLM입력 설계"  # 다른 관점으로 확장
]

print("\n=== 확장된 쿼리들로 검색 ===")
all_results = {}
for q in expanded_queries:
    results = search_faiss(q, top_k=3)
    print(f"  {q} -> Top-1: [{results[0][0]}] {documents[results[0][0]][:30]}")
    for idx, score in results:
        if idx not in all_results or score > all_results[idx]:
            all_results[idx] = score

# 합친 결과
ranked = sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:3]
print("\n=== 확장 후 합산 결과 ===")
for idx, score in ranked:
    print(f"  [{idx}] {score:.4f} | {documents[idx][:40]}")

In [ ]:
# 쿼리 확장 + RRF 합산 함수
def search_with_expansion(original, expanded_list, top_k=5):
    """확장된 쿼리로 검색하고 최고 점수만 유지"""
    all_results = {}
    for q in [original] + expanded_list:
        for idx, score in search_faiss(q, top_k=top_k):
            if idx not in all_results or score > all_results[idx]:
                all_results[idx] = score
    return sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:top_k]

def search_with_rrf(queries, top_k=3, k=60):
    """여러 쿼리 결과를 RRF로 합산"""
    rrf_scores = {}
    for query in queries:
        results = search_faiss(query, top_k=5)
        for rank, (doc_id, _) in enumerate(results, 1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

# 확장 전후 비교
original = "AI 챗봇 답변 품질 높이기"
expanded = ["RAG 검색 증강 생성 기법", "프롬프트 엔지니어링 LLM입력 설계"]

without = search_faiss(original, top_k=5)
with_exp = search_with_expansion(original, expanded, top_k=5)

without_ids = set(idx for idx, _ in without)
with_exp_ids = set(idx for idx, _ in with_exp)
new_docs = with_exp_ids - without_ids

print(f"확장 전 문서: {sorted(without_ids)}")
print(f"확장 후 문서: {sorted(with_exp_ids)}")
print(f"새로 발견된 문서: {sorted(new_docs)}")

In [ ]:
# RRF로 합산한 결과
queries = [original] + expanded
print("=== Query Expansion + RRF ===")
for idx, score in search_with_rrf(queries):
    print(f"  [{idx}] {score:.6f} | {documents[idx][:40]}")

In [ ]:
# LLM으로 자동 쿼리 확장 (Multi-Query)
llm = ChatOpenAI(model='gpt-4o-mini')

MULTI_QUERY_PROMPT = """주어진 질문을 3가지 서로 다른 관점에서 재작성하세요.
각 쿼리는 원래 질문의 의도를 유지하되, 다른 단어와 표현을 사용하세요.

원래 질문 : {query}

재작성1:
재작성2:
재작성3:"""

def generate_multi_queries(query, n=3):
    """LLM에게 쿼리를 n개의 다른 관점으로 재작성 요청"""
    response = llm.invoke(MULTI_QUERY_PROMPT.format(query=query)).content
    # 줄바꿈으로 분리하고 빈 줄 제거
    queries = [line.strip() for line in response.strip().split('\n') if line.strip()]
    return queries[:n]

# 테스트
query = '벡터검색'
expanded = generate_multi_queries(query, n=3)
print(f"원래 쿼리: {query}")
print(f"확장된 쿼리: {expanded}")

In [ ]:
def multi_query_search(query, top_k=3):
    """Multi-Query: LLM으로 쿼리 확장 -> RRF로 합산"""
    expanded = generate_multi_queries(query, n=3)
    all_queries = [query] + expanded  # 원래 쿼리 + 확장 쿼리
    return search_with_rrf(all_queries, top_k=top_k)

# Multi-Query vs 단순 검색 비교
query = '벡터검색'
print("=== Multi-Query 검색 ===")
results = multi_query_search(query, top_k=3)
for idx, score in results:
    print(f"  [{idx}] {score:.6f} | {documents[idx][:40]}")

print("\n=== 단순 FAISS 검색 ===")
results = search_faiss(query, top_k=3)
for idx, score in results:
    print(f"  [{idx}] {score:.4f} | {documents[idx][:40]}")

### 12-2. Query Diversity 측정

확장된 쿼리들이 얼마나 **다양한지** 수치로 측정한다.  
모든 쿼리 쌍의 코사인 유사도 평균을 구하고, 1에서 빼면 다양성 지표가 된다.  

- 다양성이 높다 = 서로 다른 관점에서 질문하고 있다 (좋음)  
- 다양성이 낮다 = 비슷비슷한 질문 (노이즈만 늘어남)

In [ ]:
def query_diversity(queries):
    """쿼리 리스트의 다양성 측정 (1 - 평균 코사인 유사도)"""
    if len(queries) < 2:
        return 0

    # 모든 쿼리를 임베딩으로 변환
    embs = np.array(embeddings.embed_documents(queries))  # (n, 1536)
    # L2 정규화 (코사인 유사도 계산을 단순화)
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    embs_norm = embs / norms

    # 모든 쌍의 코사인 유사도 합산
    total_sim, count = 0, 0
    for i in range(len(queries)):
        for j in range(i + 1, len(queries)):
            total_sim += np.dot(embs_norm[i], embs_norm[j])
            count += 1

    avg_sim = total_sim / count
    diversity = 1 - avg_sim  # 유사도가 낮을수록 다양성이 높다
    return diversity

# 테스트: 다양한 쿼리 vs 비슷한 쿼리
diverse = ["Python 웹 개발", "딥러닝 모델 구조", "데이터베이스 설계"]
similar = ["Python 웹 개발", "Python 웹 프레임워크", "Python 웹 서버"]

print(f"다양한 쿼리 다양성: {query_diversity(diverse):.4f}")
print(f"비슷한 쿼리 다양성: {query_diversity(similar):.4f}")
# 다양한 쿼리 세트의 diversity가 훨씬 높다

### 12-3. HyDE (Hypothetical Document Embeddings)

2023년 ACL 논문에서 발표된 기법.  
핵심: 질문 -> LLM이 **가상 답변** 생성 -> 가상 답변의 임베딩으로 검색  

왜 효과적인가?  
- 사용자 질문은 짧고 키워드 중심 ("RAG란 뭔가요?")  
- 벡터 DB에 저장된 문서는 길고 서술적  
- 가상 답변은 문서와 **형태가 비슷**해서 임베딩 공간에서 더 가까움

In [ ]:
HYDE_PROMPT = """아래 질문에 대해 3~4문장으로 답변을 작성해주세요.
정확하지 않아도 괜찮습니다. 관련 주제의 문서처럼 작성하세요.

질문 {query}

답변:"""

def generate_hypothesis(query):
    """질문에 대한 가상 답변 생성 (정확하지 않아도 됨)"""
    return llm.invoke(HYDE_PROMPT.format(query=query)).content

# 테스트: RAG에 대한 가상 답변
query = 'RAG란 무엇인가요?'
hypothesis = generate_hypothesis(query)
print(f"질문: {query}")
print(f"\n가상 답변:\n{hypothesis}")

In [ ]:
def hyde_search(query, top_k=3):
    """HyDE: 가상 답변의 임베딩으로 검색 (원래 질문 대신 답변을 검색에 사용)"""
    hypothesis = generate_hypothesis(query)
    # 가상 답변을 임베딩 -> 이 벡터로 문서 검색
    hyp_emb = embeddings.embed_query(hypothesis)
    return search_by_vector(hyp_emb, top_k)

# HyDE vs 일반 검색 비교
query = 'RAG란 무엇인가요?'

print("=== HyDE 검색 ===")
results = hyde_search(query)
for idx, score in results:
    print(f"  [{idx}] {score:.4f} | {documents[idx][:40]}")

print("\n=== 일반 FAISS 검색 ===")
results = search_faiss(query)
for idx, score in results:
    print(f"  [{idx}] {score:.4f} | {documents[idx][:40]}")

# HyDE가 RAG 문서(idx=4)를 더 높은 점수로 찾는지 확인
# (짧은 단답형 문서에서는 차이가 작지만, 긴 chunk 기반 문서에서 효과가 두드러짐)

## 정리

| 기법 | 장점 | 단점 |
|------|------|------|
| 키워드 검색 | 정확한 단어 매칭 | 동의어/유사어 못 찾음 |
| 벡터 검색 | 의미적 유사도 | 키워드 정확도 부족할 수 있음 |
| Simple Hybrid | 둘의 장점 합산 | 점수 스케일 문제 |
| RRF | 스케일 무관, 이상치에 robust | k 파라미터 튜닝 필요 |
| Multi-Query | 다양한 관점 커버 | LLM API 비용 |
| HyDE | 문서 형태에 맞는 검색 | 가상 답변 품질에 의존 |

실무에서는 이 기법들을 **조합**해서 사용하고, Grid Search로 최적 파라미터를 찾는다.